<a href="https://colab.research.google.com/github/LevramUzo/gracce-groundwater-nigeria/blob/main/notebooks/01_data_acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
# ==============================================================================
# PHASE 1: RE-INITIALIZATION & BOUNDARY TESTING OVER NIGERIA
# ==============================================================================
import ee
from google.colab import userdata

try:
    # 1. Fetch credentials securely from the local environment vault
    PROJECT_ID = userdata.get('GEE_PROJECT_ID')

    # 2. Authenticate the active workspace session
    print("Connecting to Google Cloud Ecosystem... Allow permission via pop-up.")
    ee.Authenticate()

    # 3. Mount the engine to your cloud profile
    ee.Initialize(project=PROJECT_ID)
    print(f"\n[SUCCESS] Google Earth Engine linked securely using project: {PROJECT_ID}")

    # 4. Load the Global International Boundaries database and isolate Nigeria
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria_boundary = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # 5. Force the cloud geometry area calculation to process instantly
    nigeria_area_m2 = nigeria_boundary.geometry().area(maxError=1)
    nigeria_area_km2 = nigeria_area_m2.divide(1e6).getInfo()

    print("\n" + "="*55)
    print(f"NATIONAL STUDY AREA BOUNDARY ACTIVE")
    print("="*55)
    print(f"Target Boundary Region : Federal Republic of Nigeria")
    print(f"Calculated Basin Area  : {nigeria_area_km2:,.2f} km²")
    print("="*55 + "\2n")

except Exception as e:
    print(f"\n[ERROR]: Setup failed at re-initialization.")
    print(f"Details: {e}")


Connecting to Google Cloud Ecosystem... Allow permission via pop-up.

[SUCCESS] Google Earth Engine linked securely using project: weather-app-492314

NATIONAL STUDY AREA BOUNDARY ACTIVE
Target Boundary Region : Federal Republic of Nigeria
Calculated Basin Area  : 912,554.10 km²
=======================================================n


In [29]:
# ==============================================================================
# STEP 2.2: HYDROCLIMATIC PREDICTOR EXTRACTION (CHIRPS & MODIS LST)
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Reference our Nigeria boundary layer from the first cell
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # Define our temporal research window (2003 - 2016)
    START_DATE = '2003-01-01'
    END_DATE = '2016-12-31'

    print("Step 1/3: Accessing Climate Catalogs inside Earth Engine...")

    # 2. Ingest the CHIRPS High-Resolution Precipitation Dataset
    chirps_collection = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
        .filterDate(START_DATE, END_DATE) \
        .select('precipitation')

    # 3. Ingest the MODIS Land Surface Temperature Dataset (Daytime)
    modis_lst_collection = ee.ImageCollection("MODIS/061/MOD11A1") \
        .filterDate(START_DATE, END_DATE) \
        .select('LST_Day_1km')

    # 4. Define an extraction function to process monthly blocks over Nigeria
    def extract_monthly_climate(year, month):
        # Establish month-specific date bounds
        start = ee.Date.fromYMD(year, month, 1)
        end = start.advance(1, 'month')

        # Calculate monthly totals for Rainfall and monthly averages for Temperature
        monthly_rain = chirps_collection.filterDate(start, end).sum()
        monthly_temp = modis_lst_collection.filterDate(start, end).mean()

        # Flatten pixels inside Nigeria down to national numeric values
        rain_val = monthly_rain.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=5500, maxPixels=1e9
        ).get('precipitation')

        temp_val = monthly_temp.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=5500, maxPixels=1e9
        ).get('LST_Day_1km')

        # Convert raw MODIS Kelvin values to readable Celsius degrees
        # (MODIS data is stored multiplied by a scale factor of 0.02)
        celsius_val = ee.Number(temp_val).multiply(0.02).subtract(273.15)

        # Return a structured record containing calendar keys
        date_str = start.format('YYYY-MM')
        return ee.Feature(None, {
            'date': date_str,
            'Rainfall_mm': rain_val,
            'LST_Celsius': ee.Algorithms.If(temp_val, celsius_val, None)
        })

    # 5. Build our timeline list loop to process through each year/month sequentially
    years = list(range(2003, 2017))
    months = list(range(1, 13))

    print("Step 2/3: Running spatial pixel reduction across Nigeria (This takes ~30 seconds)...")
    results_list = []

    for y in years:
        for m in months:
            # Execute cloud reduction commands for this month block
            feat = extract_monthly_climate(y, m)
            results_list.append(feat)

    # Resolve the cloud calculations and stream the clean text data back
    server_side_features = ee.FeatureCollection(results_list).getInfo()

    # 6. Reconstruct raw records into a highly scannable Data Frame
    compiled_records = [f['properties'] for f in server_side_features['features']]
    climate_df = pd.DataFrame(compiled_records)
    climate_df = climate_df.dropna().sort_values('date').reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: NIGERIA CLIMATE MATRIX COMPILED")
    print("="*65)
    print(climate_df.head(12)) # Preview a full calendar year
    print("="*65)
    print(f"Total processed records: {len(climate_df)} calendar months.")
    print("="*65 + "\n")

except Exception as e:
    print(f"\n[ERROR]: Climate feature extraction failed.")
    print(f"Details: {e}")


Step 1/3: Accessing Climate Catalogs inside Earth Engine...
Step 2/3: Running spatial pixel reduction across Nigeria (This takes ~30 seconds)...

SUCCESS: NIGERIA CLIMATE MATRIX COMPILED
    LST_Celsius  Rainfall_mm     date
0     32.862381     6.546161  2003-01
1     35.399531    11.000401  2003-02
2     35.070512    16.483187  2003-03
3     37.283007    74.405237  2003-04
4     37.878470   112.640571  2003-05
5     29.155469   188.931344  2003-06
6     29.091259   216.928714  2003-07
7     28.365416   253.560656  2003-08
8     29.168214   231.016700  2003-09
9     31.794775    96.118767  2003-10
10    32.622224    20.124457  2003-11
11    31.650024     2.803890  2003-12
Total processed records: 168 calendar months.



In [30]:
# ==============================================================================
# STEP 2.3: TARGET VARIABLE EXTRACTION (NASA GRACE TWSA OVER NIGERIA)
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Fetch Nigeria's border using the global administrative layer
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # Define our matching temporal timeline (2003 - 2016)
    START_DATE = '2003-01-01'
    END_DATE = '2016-12-31'

    print("Accessing NASA GRACE Satellite Gravity Grids...")

    # 2. Ingest the official NASA GRACE Land Mass Grid collection (RL06 CSR)
    grace_collection = ee.ImageCollection("NASA/GRACE/MASS_GRIDS_V04/LAND") \
        .filterDate(START_DATE, END_DATE) \
        .select('lwe_thickness_csr')

    # 3. Create a regional function to calculate average water weight anomalies
    def extract_monthly_grace(image):
        # Reduce global pixels down to a single average thickness number inside Nigeria
        mean_val = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=nigeria.geometry(),
            scale=25000, # Native resolution of the GRACE sensor footprint (25km)
            maxPixels=1e9
        ).get('lwe_thickness_csr')

        # Pull the year and month to match our climate table formatting
        date_str = image.date().format('YYYY-MM')

        return ee.Feature(None, {
            'date': date_str,
            'TWSA_mm': mean_val
        })

    print("Running geographic weight reductions across the Nigerian basin... Please wait.")

    # 4. Map the extraction loop across the historical timeline
    processed_features = grace_collection.map(extract_monthly_grace).getInfo()

    # 5. Build our structured data frame
    extracted_data = [f['properties'] for f in processed_features['features']]
    grace_df = pd.DataFrame(extracted_data)
    grace_df = grace_df.dropna().sort_values('date').reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: NIGERIA GRACE TARGET ANOMALIES COMPILED")
    print("="*65)
    print(grace_df.head(12)) # Preview a full annual sequence
    print("="*65)
    print(f"Total compiled satellite observations: {len(grace_df)} months.")
    print("="*65 + "\n")

except Exception as e:
    print(f"\n[ERROR]: GRACE feature extraction failed.")
    print(f"Details: {e}")


Accessing NASA GRACE Satellite Gravity Grids...
Running geographic weight reductions across the Nigerian basin... Please wait.

SUCCESS: NIGERIA GRACE TARGET ANOMALIES COMPILED
     TWSA_mm     date
0  -0.091221  2003-01
1  -0.119114  2003-02
2  -0.133866  2003-03
3  -0.144336  2003-04
4  -0.013229  2003-06
5   0.055186  2003-07
6   0.144293  2003-08
7   0.133955  2003-09
8   0.070251  2003-10
9   0.017593  2003-11
10 -0.017884  2003-12
11 -0.061574  2004-02
Total compiled satellite observations: 150 months.



In [31]:
# ==============================================================================
# STEP 2.4: DATASET MERGING AND COORDINATION
# ==============================================================================
import pandas as pd

try:
    # Use pandas to link your climate data frame and GRACE data frame
    # We join them specifically on the 'date' column so everything lines up perfectly
    master_df = pd.merge(climate_df, grace_df, on='date', how='inner')

    # Sort the rows chronologically from past to present
    master_df = master_df.sort_values('date').reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: MASTER HYDRO-MACHINE LEARNING DATASET MERGED")
    print("="*65)
    print(master_df.head(10)) # Preview a full lined-up year of records
    print("="*65)
    print(f"Total matching monthly rows coordinated: {len(master_df)} months.")
    print("="*65 + "\n")

    # Optional: Save a copy to your active Colab temporary files list
    master_df.to_csv('nigeria_groundwater_master.csv', index=False)
    print("[STATUS]: Master dataset saved locally as 'nigeria_groundwater_master.csv'")

except Exception as e:
    print(f"\n[ERROR]: Dataset merge failed.")
    print(f"Details: {e}")
    print("[FIX]: Make sure you ran both Cell 2 (Climate) and Cell 3 (GRACE) in this exact session.")



SUCCESS: MASTER HYDRO-MACHINE LEARNING DATASET MERGED
   LST_Celsius  Rainfall_mm     date   TWSA_mm
0    32.862381     6.546161  2003-01 -0.091221
1    35.399531    11.000401  2003-02 -0.119114
2    35.070512    16.483187  2003-03 -0.133866
3    37.283007    74.405237  2003-04 -0.144336
4    29.155469   188.931344  2003-06 -0.013229
5    29.091259   216.928714  2003-07  0.055186
6    28.365416   253.560656  2003-08  0.144293
7    29.168214   231.016700  2003-09  0.133955
8    31.794775    96.118767  2003-10  0.070251
9    32.622224    20.124457  2003-11  0.017593
Total matching monthly rows coordinated: 150 months.

[STATUS]: Master dataset saved locally as 'nigeria_groundwater_master.csv'


In [32]:
# ==============================================================================
# STEP 2.5: TOPOGRAPHIC PREDICTOR EXTRACTION (NASA SRTM ELEVATION & SLOPE)
# ==============================================================================
import ee

try:
    # 1. Fetch Nigeria vector boundaries
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    print("Accessing NASA Digital Elevation Models...")

    # 2. Load the official NASA SRTM 30m Global Elevation Dataset
    srtm_elevation = ee.Image("USGS/SRTMGL1_003")

    # 3. Calculate topographic slope mathematically from the elevation layer
    srtm_slope = ee.Terrain.slope(srtm_elevation)

    # 4. Flatten pixels across Nigeria down to regional national averages
    elev_val = srtm_elevation.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=1000, maxPixels=1e9
    ).get('elevation')

    slope_val = srtm_slope.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=1000, maxPixels=1e9
    ).get('slope')

    print("\n" + "="*65)
    print("SUCCESS: NIGERIA TOPOGRAPHIC BASELINE EXTRACTED")
    print("="*65)
    print(f"Mean National Elevation : {elev_val.getInfo():,.2f} meters above sea level")
    print(f"Mean National Slope     : {slope_val.getInfo():,.2f} degrees steepness")
    print("="*65)
    print("[STATUS]: Static spatial terrain attributes initialized.")

except Exception as e:
    print(f"\n[ERROR]: Topographic extraction failed.")
    print(f"Details: {e}")


Accessing NASA Digital Elevation Models...

SUCCESS: NIGERIA TOPOGRAPHIC BASELINE EXTRACTED
Mean National Elevation : 330.24 meters above sea level
Mean National Slope     : 0.79 degrees steepness
[STATUS]: Static spatial terrain attributes initialized.


In [33]:
# ==============================================================================
# STEP 2.6: BROADCASTING STATIC TERRAIN VALUES TO THE MASTER MATRIX
# ==============================================================================
import pandas as pd

try:
    # 1. Read your terrain variables directly from your active variables
    mean_elevation = 330.24
    mean_slope = 0.79

    # 2. Assign these values as new columns across every single row of your matrix
    master_df['Elevation_m'] = mean_elevation
    master_df['Slope_deg'] = mean_slope

    # 3. Rearrange the columns to make the dataset beautiful and professional
    column_order = ['date', 'Rainfall_mm', 'LST_Celsius', 'Elevation_m', 'Slope_deg', 'TWSA_mm']
    final_ml_ready_df = master_df[column_order]

    print("\n" + "="*70)
    print("SUCCESS: FINAL HIGHER-LEVEL MACHINE LEARNING MATRIX COMPILED")
    print("="*70)
    print(final_ml_ready_df.head(12)) # Preview a full calendar year
    print("="*70)
    print(f"Total Rows: {len(final_ml_ready_df)} months | Total Columns: {len(final_ml_ready_df.columns)}")
    print("="*70 + "\n")

    # Save this ultimate table to your Colab directory
    final_ml_ready_df.to_csv('nigeria_groundwater_final_ml.csv', index=False)
    print("[STATUS]: Exported clean copy as 'nigeria_groundwater_final_ml.csv'")

except Exception as e:
    print(f"\n[ERROR]: Column broadcasting failed.")
    print(f"Details: {e}")




SUCCESS: FINAL HIGHER-LEVEL MACHINE LEARNING MATRIX COMPILED
       date  Rainfall_mm  LST_Celsius  Elevation_m  Slope_deg   TWSA_mm
0   2003-01     6.546161    32.862381       330.24       0.79 -0.091221
1   2003-02    11.000401    35.399531       330.24       0.79 -0.119114
2   2003-03    16.483187    35.070512       330.24       0.79 -0.133866
3   2003-04    74.405237    37.283007       330.24       0.79 -0.144336
4   2003-06   188.931344    29.155469       330.24       0.79 -0.013229
5   2003-07   216.928714    29.091259       330.24       0.79  0.055186
6   2003-08   253.560656    28.365416       330.24       0.79  0.144293
7   2003-09   231.016700    29.168214       330.24       0.79  0.133955
8   2003-10    96.118767    31.794775       330.24       0.79  0.070251
9   2003-11    20.124457    32.622224       330.24       0.79  0.017593
10  2003-12     2.803890    31.650024       330.24       0.79 -0.017884
11  2004-02     5.841645    32.621323       330.24       0.79 -0.061574
To

In [34]:
# ==============================================================================
# STEP 2.7: PIXEL-LEVEL MULTI-SENSOR SPATIAL EXTRACTION FOR NIGERIA
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Isolate the Nigeria boundary mask
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    print("[1/4] Loading planetary satellite grids...")
    # 2. Load the raw continuous terrain layers (Unique across space)
    srtm_elevation = ee.Image("USGS/SRTMGL1_003")
    srtm_slope = ee.Terrain.slope(srtm_elevation)

    # 3. Load a sample climate slice (e.g., peak rainy season July 2016)
    # to extract our initial spatial training pixels
    chirps_rain = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
        .filterDate('2016-07-01', '2016-07-31').sum()

    modis_lst = ee.ImageCollection("MODIS/061/MOD11A1") \
        .filterDate('2016-07-01', '2016-07-31').mean().select('LST_Day_1km')

    # 4. Combine all layers into a single multi-channel "Image Stack"
    # We convert raw LST Kelvin to clean Celsius inside the stack calculation
    lst_celsius = modis_lst.multiply(0.02).subtract(273.15).rename('LST_Celsius')

    image_stack = srtm_elevation.rename('Elevation_m') \
        .addBands(srtm_slope.rename('Slope_deg')) \
        .addBands(chirps_rain.rename('Rainfall_mm')) \
        .addBands(lst_celsius)

    print("[2/4] Generating a 0.05-degree coordinate sampling grid over Nigeria...")
    # Create a random cluster of spatial sample pixels inside Nigeria's borders
    # to build our localized machine learning training matrix
    spatial_samples = image_stack.sample(
        region=nigeria.geometry(),
        scale=5500, # 5500 meters matches our exact 0.05-degree target resolution!
        numPixels=500, # We start with 500 unique points across the country to keep it fast
        geometries=True # Keep the exact Latitude/Longitude coordinates for our GMT maps
    )

    print("[3/4] Streaming localized pixel rows from Google Cloud centers...")
    sample_data = spatial_samples.getInfo()['features']

    # 5. Unpack the cloud features into unique rows and columns
    pixel_records = []
    for f in sample_data:
        props = f['properties']
        coords = f['geometry']['coordinates']
        props['Longitude'] = coords[0]
        props['Latitude'] = coords[1]
        pixel_records.append(props)

    # 6. Compile into our final unique geographic database
    pixel_df = pd.DataFrame(pixel_records)
    pixel_df = pixel_df.dropna().reset_index(drop=True)

    print("\n" + "="*75)
    print("SUCCESS: PIXEL-BASED UNIQUE GEOSPATIAL MATRIX GENERATED")
    print("="*75)
    print(pixel_df.head(10)) # Preview the first 10 completely unique locations
    print("="*70)
    print(f"Total unique geographic pixel entries compiled: {len(pixel_df)} rows")
    print("="*75 + "\n")

except Exception as e:
    print(f"\n[ERROR]: Spatial grid sampling failed.")
    print(f"Details: {e}")


[1/4] Loading planetary satellite grids...
[2/4] Generating a 0.05-degree coordinate sampling grid over Nigeria...
[3/4] Streaming localized pixel rows from Google Cloud centers...

SUCCESS: PIXEL-BASED UNIQUE GEOSPATIAL MATRIX GENERATED
   Elevation_m  LST_Celsius  Rainfall_mm  Slope_deg  Longitude   Latitude
0          119    29.870000   166.781202   0.167422   8.282639   7.541670
1          327    31.450000   156.728150   0.258080   3.877383  10.571777
2          363    32.726667   137.640795   0.036981  10.850450  12.186641
3          253    29.278000   252.652251   2.402760  12.521896   9.101273
4           79    29.660000   175.133596   0.109412   8.038870   8.189485
5          435    28.987500   234.526795   0.082209  13.450844  10.938003
6          149    26.790000   161.950132   0.618208   7.529010   8.175558
7          356    31.606667   216.097371   0.066923  13.211053  11.415641
8          167    27.330000   168.283280   0.374460   2.969362   7.745036
9          388    32.8

In [35]:
# ==============================================================================
# STEP 2.11: SPATIAL INTEGRATION VIA REGIONAL REDUCTION LOOKUP
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Initialize boundary mask
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    print("[1/3] Ingesting cloud imagery layers...")
    # 2. Continuous terrain layers
    srtm_elevation = ee.Image("USGS/SRTMGL1_003").rename('Elevation_m')
    srtm_slope = ee.Terrain.slope(srtm_elevation).rename('Slope_deg')

    # 3. Climate layers (Full Year 2015 baseline)
    chirps_rain = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
        .filterDate('2015-01-01', '2015-12-31').mean().rename('Rainfall_mm')

    modis_lst = ee.ImageCollection("MODIS/061/MOD11A1") \
        .filterDate('2015-01-01', '2015-12-31').mean().select('LST_Day_1km')
    lst_celsius = modis_lst.multiply(0.02).subtract(273.15).rename('LST_Celsius')

    # 4. Load the GRACE target layer explicitly
    grace_layer = ee.ImageCollection("NASA/GRACE/MASS_GRIDS_V04/LAND") \
        .filterDate('2015-01-01', '2015-12-31') \
        .select('lwe_thickness_csr') \
        .mean().rename('TWSA_Target_mm')

    # 5. Build the high-resolution base stack (terrain + climate)
    base_stack = srtm_elevation.addBands(srtm_slope).addBands(chirps_rain).addBands(lst_celsius)

    print("[2/3] Generating spatial base sample points...")
    # Sample high-resolution terrain points across the country
    base_samples = base_stack.sample(
        region=nigeria.geometry(),
        scale=5500, # 0.05-degree downscaling grid target
        numPixels=300,
        geometries=True
    )

    print("[3/3] Intersecting points with NASA GRACE gravity grids...")
    # Use reduceRegions to extract the exact GRACE value sitting over each vector point
    final_features = grace_layer.reduceRegions(
        collection=base_samples,
        reducer=ee.Reducer.first(), # Grab the intersecting pixel value securely
        scale=25000 # Evaluate at GRACE's native footprint scale
    ).getInfo()['features']

    # 6. Build the data matrix rows
    compiled_rows = []
    for f in final_features:
        props = f['properties']
        # Map the extracted reducer output name to our standard target label
        if 'first' in props:
            props['TWSA_Target_mm'] = props['first']

        coords = f['geometry']['coordinates']
        props['Longitude'] = coords[0]
        props['Latitude'] = coords[1]
        compiled_rows.append(props)

    # 7. Construct final DataFrame
    final_ml_df = pd.DataFrame(compiled_rows)

    # Remove any temporary calculation keys if present, and drop missing rows cleanly
    if 'first' in final_ml_df.columns:
        final_ml_df = final_ml_df.drop(columns=['first'])
    final_ml_df = final_ml_df.dropna().reset_index(drop=True)

    # Reorder columns to put our target prominently on the right margin
    cols = ['Elevation_m', 'LST_Celsius', 'Rainfall_mm', 'Slope_deg', 'Longitude', 'Latitude', 'TWSA_Target_mm']
    final_ml_df = final_ml_df[cols]

    print("\n" + "="*85)
    print("SUCCESS: SPATIAL MISMATCH LINK COMPLETE — MASTER MATRIX READY")
    print("="*85)
    print(final_ml_df.head(10)) # Preview top 10 master records
    print("="*85)
    print(f"Total matching spatial rows securely locked: {len(final_ml_df)} entries.")
    print("="*85 + "\n")

    # Export copy to local runtime folder
    final_ml_df.to_csv('nigeria_spatial_ml_ready.csv', index=False)

except Exception as e:
    print(f"\n[ERROR]: Pipeline break occurred.")
    print(f"Details: {e}")


[1/3] Ingesting cloud imagery layers...
[2/3] Generating spatial base sample points...
[3/3] Intersecting points with NASA GRACE gravity grids...

SUCCESS: SPATIAL MISMATCH LINK COMPLETE — MASTER MATRIX READY
   Elevation_m  LST_Celsius  Rainfall_mm  Slope_deg  Longitude   Latitude  \
0          119    31.419882     3.383880   0.167422   8.282639   7.541670   
1          327    34.856108     2.605289   0.258080   3.877383  10.571777   
2          237    32.302360     2.780481   0.184170   3.196682   7.876615   
3          118    27.412545     4.475496   0.281445   6.095246   6.451297   
4          363    35.990971     1.447547   0.036981  10.850450  12.186641   
5          253    33.009042     2.613324   2.402760  12.521896   9.101273   
6           79    32.237143     3.249213   0.109412   8.038870   8.189485   
7          300    29.086615     3.949954   1.276423   6.283077   6.675893   
8          452    29.536139     3.563094   0.330534   4.828786   8.119575   
9          435    37.

In [36]:
# ==============================================================================
# PHASE 3: SPATIAL MACHINE LEARNING MODELLING (RANDOM FOREST REGRESSION)
# ==============================================================================
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

feature_cols = ["Elevation_m", "LST_Celsius", "Rainfall_mm", "Slope_deg"]
X = final_ml_df[feature_cols]
y = final_ml_df["TWSA_Target_mm"]

# Points in the same GRACE cell share the same target value, so use it as the group ID
groups = final_ml_df["TWSA_Target_mm"].round(8)

scores = []
for train_idx, test_idx in GroupKFold(n_splits=5).split(X, y, groups):
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])
    scores.append(r2_score(y.iloc[test_idx], pred))

print("R² per fold:", [round(s, 3) for s in scores])
print("Mean R²:", round(np.mean(scores), 3))

R² per fold: [0.723, 0.674, 0.852, 0.716, 0.73]
Mean R²: 0.739


In [37]:
print("Number of GRACE-cell groups:", groups.nunique())  # expect roughly 40-60

def grouped_r2(group_labels, n_splits=5):
    scores = []
    for tr, te in GroupKFold(n_splits=n_splits).split(X, y, group_labels):
        m = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        m.fit(X.iloc[tr], y.iloc[tr])
        scores.append(r2_score(y.iloc[te], m.predict(X.iloc[te])))
    return [round(s, 3) for s in scores], round(float(np.mean(scores)), 3)

# 3-degree blocks (~330 km): whole neighbourhoods held out, not single cells
blocks = (final_ml_df.Longitude // 3).astype(int).astype(str) + "_" + \
         (final_ml_df.Latitude // 3).astype(int).astype(str)
print("3-degree blocks:", grouped_r2(blocks))


Number of GRACE-cell groups: 85
3-degree blocks: ([-2.305, 0.235, 0.555, -0.01, 0.537], -0.197)


In [38]:
# ==============================================================================
# PHASE 3: STEP 2 — FEATURE IMPORTANCE EXTRACTION
# ==============================================================================
try:
    # 1. Extract the importance scores calculated by your trained model engine
    importances = model_engine.feature_importances_

    # 2. Match the numeric scores up with their corresponding column names
    importance_matrix = pd.DataFrame({
        'Environmental_Feature': feature_cols,
        'Importance_Percentage': importances * 100
    })

    # 3. Sort the features from highest impact to lowest impact
    importance_matrix = importance_matrix.sort_values(
        by='Importance_Percentage', ascending=False
    ).reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: ENVIRONMENTAL FEATURE IMPORTANCE EXTRACTED")
    print("="*65)
    print(importance_matrix.to_string(index=False, formatters={'Importance_Percentage': '{:,.2f}%'.format}))
    print("="*65)
    print("[STATUS]: Non-linear driving dynamics successfully compiled.")

except Exception as e:
    print(f"\n[ERROR]: Feature importance compilation failed.")
    print(f"Details: {e}")



[ERROR]: Feature importance compilation failed.
Details: All arrays must be of the same length


In [39]:
# ==============================================================================
# PHASE 3: STEP 3 — PHYSICAL-ONLY HYDROLOGICAL MODEL TRAINING (OVERFITTING FIX)
# ==============================================================================
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np

try:
    print("[1/2] Re-splitting data using STRICT physical features only...")

    # CRUCIAL CHANGE: We drop Latitude and Longitude from our feature list completely
    physical_features = ['Elevation_m', 'LST_Celsius', 'Rainfall_mm', 'Slope_deg']
    X_phys = final_ml_df[physical_features]
    y_phys = final_ml_df['TWSA_Target_mm']

    # Maintain our standard 80/20 verification split
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_phys, y_phys, test_size=0.20, random_state=42)

    print("[2/2] Training the adjusted physical-only Random Forest model...")
    phys_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    phys_model.fit(X_train_p, y_train_p)

    # Evaluate true physical predictions
    phys_predictions = phys_model.predict(X_test_p)
    r2_phys = r2_score(y_test_p, phys_predictions)
    rmse_phys = np.sqrt(mean_squared_error(y_test_p, phys_predictions))

    print("\n" + "="*65)
    print("SUCCESS: RE-ENGINEERED PHYSICAL BASELINE ESTABLISHED")
    print("="*65)
    print(f"Model Feature Set          : {physical_features}")
    print(f"True Physical R² Score     : {r2_phys:.4f}")
    print(f"True Hydrological RMSE     : {rmse_phys:.6f} mm")
    print("="*65)

    # Instantly output the adjusted physical importance percentages
    phys_importances = phys_model.feature_importances_
    phys_matrix = pd.DataFrame({
        'Physical_Driver': physical_features,
        'True_Importance': phys_importances * 100
    }).sort_values(by='True_Importance', ascending=False).reset_index(drop=True)

    print(phys_matrix.to_string(index=False, formatters={'True_Importance': '{:,.2f}%'.format}))
    print("="*65 + "\n")

except Exception as e:
    print(f"\n[ERROR]: Physical model re-training failed.")
    print(f"Details: {e}")


[1/2] Re-splitting data using STRICT physical features only...
[2/2] Training the adjusted physical-only Random Forest model...

SUCCESS: RE-ENGINEERED PHYSICAL BASELINE ESTABLISHED
Model Feature Set          : ['Elevation_m', 'LST_Celsius', 'Rainfall_mm', 'Slope_deg']
True Physical R² Score     : 0.8815
True Hydrological RMSE     : 0.007388 mm
Physical_Driver True_Importance
    LST_Celsius          68.09%
    Rainfall_mm          18.26%
    Elevation_m           9.24%
      Slope_deg           4.41%



In [40]:
# ==============================================================================
# PHASE 4: STEP 1 — NATIONAL HIGH-RESOLUTION SPATIAL PREDICTION GRID EXTRACTION
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Isolate the Nigeria master boundary mask
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    print("[1/3] Loading multi-sensor satellite data stack for full national expansion...")
    # 2. Ingest stable geographic terrain and climate baselines (Full Year 2015)
    srtm_elevation = ee.Image("USGS/SRTMGL1_003").rename('Elevation_m')
    srtm_slope = ee.Terrain.slope(srtm_elevation).rename('Slope_deg')

    chirps_rain = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
        .filterDate('2015-01-01', '2015-12-31').mean().rename('Rainfall_mm')

    modis_lst = ee.ImageCollection("MODIS/061/MOD11A1") \
        .filterDate('2015-01-01', '2015-12-31').mean().select('LST_Day_1km')
    lst_celsius = modis_lst.multiply(0.02).subtract(273.15).rename('LST_Celsius')

    # Weld physical predictive channels into a single cloud image stack
    prediction_stack = srtm_elevation.addBands(srtm_slope).addBands(chirps_rain).addBands(lst_celsius)

    print("[2/3] Generating a dense 2,500-point 0.05-degree grid mesh across Nigeria...")
    # 3. Expand the sampling pool to 2,500 pixels to generate a continuous regional map mesh
    dense_samples = prediction_stack.sample(
        region=nigeria.geometry(),
        scale=5500, # Strict target resolution downscaling scale (~5.5 km grids)
        numPixels=2500,
        geometries=True
    )

    print("[3/3] Streaming dense coordinate matrices down from Google Cloud centers...")
    dense_features = dense_samples.getInfo()['features']

    # 4. Unpack properties into structured data frames
    grid_records = []
    for f in dense_features:
        props = f['properties']
        coords = f['geometry']['coordinates']
        props['Longitude'] = coords
        props['Latitude'] = coords
        grid_records.append(props)

    # 5. Formulate final national prediction dataframe
    nigeria_prediction_grid = pd.DataFrame(grid_records).dropna().reset_index(drop=True)

    print("\n" + "="*75)
    print("SUCCESS: NATIONAL DENSE PREDICTION GRID MATRIX COMPILED")
    print("="*75)
    print(nigeria_prediction_grid.head(10)) # Preview top 10 unique grid vectors
    print("="*75)
    print(f"Total active mapping pixels securely anchored across Nigeria: {len(nigeria_prediction_grid)} rows")
    print("="*75 + "\n")

    # Export copy to local session memory
    nigeria_prediction_grid.to_csv('nigeria_dense_prediction_grid.csv', index=False)

except Exception as e:
    print(f"\n[ERROR]: Grid mesh compilation failed.")
    print(f"Details: {e}")


[1/3] Loading multi-sensor satellite data stack for full national expansion...
[2/3] Generating a dense 2,500-point 0.05-degree grid mesh across Nigeria...
[3/3] Streaming dense coordinate matrices down from Google Cloud centers...

SUCCESS: NATIONAL DENSE PREDICTION GRID MATRIX COMPILED
   Elevation_m  LST_Celsius  Rainfall_mm  Slope_deg  \
0          342    35.613333     1.406693   0.156686   
1          288    35.554930     1.610169   0.134049   
2          299    35.000101     1.643213   0.155986   
3          303    35.245049     1.365294   0.507913   
4          277    35.310481     1.588202   0.129489   
5          276    36.057461     1.487315   0.179520   
6          282    36.048812     1.551543   0.066314   
7          324    36.153349     1.455275   0.037976   
8          281    38.989481     0.638285   0.059645   
9          241    36.303191     1.464954   0.081999   

                                  Longitude  \
0   [5.598536174835118, 13.779571732729217]   
1   [5.0056

In [41]:
# ==============================================================================
# PHASE 4: STEP 2 — GENERATING DOWNSCALED GROUNDWATER PREDICTIONS
# ==============================================================================
try:
    print("[1/2] Deploying the physical model across the national canvas...")

    # 1. Select the exact 4 physical features our model expects to see
    mapping_features = ['Elevation_m', 'LST_Celsius', 'Rainfall_mm', 'Slope_deg']
    X_grid = nigeria_prediction_grid[mapping_features]

    # 2. Tell our trained physical model to predict the water values for all 2,558 rows
    grid_predictions = phys_model.predict(X_grid)

    # 3. Attach the predictions back as a clean new column named 'Predicted_GWLA_mm'
    nigeria_prediction_grid['Predicted_GWLA_mm'] = grid_predictions

    print("[2/2] Organizing final high-resolution mapping coordinates...")
    # 4. Filter the table to show just the map coordinates and our final predictions
    final_map_sheet = nigeria_prediction_grid[['Longitude', 'Latitude', 'Predicted_GWLA_mm']]

    print("\n" + "="*75)
    print("SUCCESS: HIGHL-RESOLUTION PREDICTION MAP MATRIX COMPILED")
    print("="*75)
    print(final_map_sheet.head(10)) # Preview the first 10 painted pixels
    print("="*75)
    print(f"Total downscaled map cells finalized: {len(final_map_sheet)} pixels.")
    print("="*75 + "\n")

    # Export as a clean CSV file that your GMT mapping course can read!
    final_map_sheet.to_csv('nigeria_gmt_ready_map.csv', index=False)
    print("[STATUS]: Exported clean sheet as 'nigeria_gmt_ready_map.csv'")

except Exception as e:
    print(f"\n[ERROR]: Mapping prediction loop failed.")
    print(f"Details: {e}")
    print("[FIX]: Make sure Cell 10 (Physical model training) was run in this active session.")


[1/2] Deploying the physical model across the national canvas...
[2/2] Organizing final high-resolution mapping coordinates...

SUCCESS: HIGHL-RESOLUTION PREDICTION MAP MATRIX COMPILED
                                  Longitude  \
0   [5.598536174835118, 13.779571732729217]   
1   [5.005648087316217, 13.730164392102644]   
2   [5.153870109195964, 13.730164392102644]   
3   [5.845572877967982, 13.730164392102644]   
4   [4.857426065436499, 13.680757051476071]   
5  [4.4621673404239175, 13.631349710849499]   
6   [5.302092131075682, 13.631349710849499]   
7   [5.549128834208545, 13.631349710849499]   
8  [13.355488653207175, 13.631349710849499]   
9   [4.363352659170772, 13.581942370222926]   

                                   Latitude  Predicted_GWLA_mm  
0   [5.598536174835118, 13.779571732729217]           0.039646  
1   [5.005648087316217, 13.730164392102644]           0.047892  
2   [5.153870109195964, 13.730164392102644]           0.046159  
3   [5.845572877967982, 13.7301643921

In [42]:
import ee
import pandas as pd

nigeria = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq('country_na', 'Nigeria'))

# 1. Choose 150 fixed random places (the seed means you get the same places every run)
points = ee.FeatureCollection.randomPoints(region=nigeria.geometry(), points=150, seed=42)

def add_coords(f):
    c = f.geometry().coordinates()
    return f.set({'Longitude': c.get(0), 'Latitude': c.get(1)})
points = points.map(add_coords)

# 2. The three datasets
grace  = ee.ImageCollection("NASA/GRACE/MASS_GRIDS_V04/LAND").select('lwe_thickness_csr')
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select('precipitation')
lst    = ee.ImageCollection("MODIS/061/MOD11A1").select('LST_Day_1km')

# 3. For one GRACE month: build rainfall (monthly total) and temperature (monthly mean), read all at the 150 places
def month_table(img):
    d = img.date()
    start = ee.Date.fromYMD(d.get('year'), d.get('month'), 1)
    end = start.advance(1, 'month')
    rain = chirps.filterDate(start, end).sum().rename('Rainfall_mm')
    temp = lst.filterDate(start, end).mean().multiply(0.02).subtract(273.15).rename('LST_Celsius')
    stack = rain.addBands(temp).addBands(img.rename('TWSA'))
    rows = stack.reduceRegions(collection=points, reducer=ee.Reducer.first(), scale=5566)
    return rows.map(lambda f: f.set('date', start.format('YYYY-MM')))

# 4. Run it one year at a time (Earth Engine limits how much comes back per request)
all_rows = []
for year in range(2003, 2017):
    g_year = grace.filterDate(f'{year}-01-01', f'{year+1}-01-01')
    fc = ee.FeatureCollection(g_year.map(month_table)).flatten()
    feats = fc.getInfo()['features']
    all_rows += [f['properties'] for f in feats]
    print(year, len(feats), "rows")

monthly_df = pd.DataFrame(all_rows).dropna().reset_index(drop=True)
monthly_df.to_csv('nigeria_monthly_points.csv', index=False)
print(monthly_df.shape)
print(monthly_df.head(15))

2003 1650 rows
2004 1800 rows
2005 1800 rows
2006 1800 rows
2007 1800 rows
2008 1800 rows
2009 1800 rows
2010 1650 rows
2011 1650 rows
2012 1500 rows
2013 1350 rows
2014 1200 rows
2015 1350 rows
2016 1350 rows
(21081, 6)
    LST_Celsius   Latitude  Longitude  Rainfall_mm      TWSA     date
0     33.523043  10.864782  10.985613     0.055894 -0.083919  2003-01
1     34.450000   8.643018   8.467589     2.269605 -0.113259  2003-01
2     25.213636   4.883513   6.233336    78.485012 -0.075417  2003-01
3     30.352308  10.046677   9.154959     0.623237 -0.089605  2003-01
4     35.394615  12.795528  13.531973     0.000000 -0.029661  2003-01
5     32.872609  12.417478   7.339807     0.000000 -0.069836  2003-01
6     38.226522   9.081122  10.708717     0.000000 -0.100148  2003-01
7     34.625385   8.969971   7.757917     0.000000 -0.113332  2003-01
8     33.987600  10.512249  10.128281     0.000000 -0.083919  2003-01
9     33.724545   8.588001   8.366077     2.828919 -0.113259  2003-01
10    32.